In [1]:
!pip install presidio-analyzer presidio-anonymizer
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

# Create Presidio engines
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

print("Presidio setup complete.")

Presidio setup complete.


In [3]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete!")

Enter your OpenAI API key: ··········
Setup complete!


In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

response = llm.invoke(
    """
    Generate a short customer order confirmation.
    Include a fictional customer name, email address, and phone number.
    """
)

agent_output = response.content

print("Agent Output:")
print(agent_output)

Agent Output:
**Order Confirmation**

Dear Sarah Thompson,

Thank you for your order! We are pleased to confirm that we have received your order and it is being processed.

**Order Details:**
- **Order Number:** 123456
- **Order Date:** October 5, 2023

**Shipping Information:**
- **Name:** Sarah Thompson
- **Email:** sarah.thompson@email.com
- **Phone:** (555) 123-4567

You will receive another email once your order has shipped. If you have any questions, feel free to reach out to our customer service team.

Thank you for choosing us!

Best regards,  
[Your Company Name]  
Customer Service Team  
[Your Company Phone Number]  
[Your Company Email Address]  


In [6]:
results = analyzer.analyze(
    text=agent_output,
    language="en",
    entities=[
        "PERSON",
        "EMAIL_ADDRESS",
        "PHONE_NUMBER"
    ]
)

print("Detected PII:")

for r in results:
    print(
        f"{r.entity_type}: "
        f"{agent_output[r.start:r.end]}"
    )

Detected PII:
EMAIL_ADDRESS: sarah.thompson@email.com
PERSON: Sarah Thompson
PERSON: Sarah Thompson
PHONE_NUMBER: (555) 123-4567


In [7]:
protected_output = anonymizer.anonymize(
    text=agent_output,
    analyzer_results=results
)

print("Protected Output:")
print(protected_output.text)

Protected Output:
**Order Confirmation**

Dear <PERSON>,

Thank you for your order! We are pleased to confirm that we have received your order and it is being processed.

**Order Details:**
- **Order Number:** 123456
- **Order Date:** October 5, 2023

**Shipping Information:**
- **Name:** <PERSON>
- **Email:** <EMAIL_ADDRESS>
- **Phone:** <PHONE_NUMBER>

You will receive another email once your order has shipped. If you have any questions, feel free to reach out to our customer service team.

Thank you for choosing us!

Best regards,  
[Your Company Name]  
Customer Service Team  
[Your Company Phone Number]  
[Your Company Email Address]  


In [8]:
def pii_output_guardrail(agent_output):

    # Detect PII
    results = analyzer.analyze(
        text=agent_output,
        language="en",
        entities=[
            "PERSON",
            "EMAIL_ADDRESS",
            "PHONE_NUMBER"
        ]
    )

    # Keep only the PII types we want to protect
    results = [
        r for r in results
        if r.entity_type in [
            "PERSON",
            "EMAIL_ADDRESS",
            "PHONE_NUMBER"
        ]
        and r.start < r.end
    ]

    # If no PII is detected, return the original output
    if not results:
        return agent_output

    # Anonymize detected PII
    protected_output = anonymizer.anonymize(
        text=agent_output,
        analyzer_results=results
    )

    return protected_output.text

In [9]:
safe_output = pii_output_guardrail(agent_output)

print("Final Safe Output:")
print(safe_output)

Final Safe Output:
**Order Confirmation**

Dear <PERSON>,

Thank you for your order! We are pleased to confirm that we have received your order and it is being processed.

**Order Details:**
- **Order Number:** 123456
- **Order Date:** October 5, 2023

**Shipping Information:**
- **Name:** <PERSON>
- **Email:** <EMAIL_ADDRESS>
- **Phone:** <PHONE_NUMBER>

You will receive another email once your order has shipped. If you have any questions, feel free to reach out to our customer service team.

Thank you for choosing us!

Best regards,  
[Your Company Name]  
Customer Service Team  
[Your Company Phone Number]  
[Your Company Email Address]  
